# Townsend avalanche current with secondary emission

**Prerequisite:** [Townsend avalanche and secondary-emission current](../../../docs/lecture-notes/plasma-physics/townsend-discharge/index.md)

This maintained laboratory replaces the notebook formerly stored inside the Python package. The state, result, closed-form evaluator, generational sum, and fixed-point iteration now belong to `projectkoios.physkit`.

## Learning objectives

1. Calculate avalanche gain and the secondary-emission feedback factor.
2. Compare closed-form, generational, and fixed-point current evaluation.
3. Identify the represented breakdown threshold $\gamma_e(M-1)=1$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.plasmas.gas_discharge.townsend import (
    TownsendClosedFormCurrentEvaluator,
    TownsendDischargeState,
    TownsendFixedPointCurrentEvaluator,
    TownsendGenerationalCurrentEvaluator,
)


In [ ]:
state = TownsendDischargeState(
    primary_current_amperes=1e-12,
    first_townsend_coefficient_per_metre=200.0,
    gap_length_metres=0.01,
    secondary_emission_coefficient=0.02,
)

closed_form = TownsendClosedFormCurrentEvaluator().execute(state)
generational = TownsendGenerationalCurrentEvaluator(
    relative_tolerance=1e-14,
).execute(state)
fixed_point = TownsendFixedPointCurrentEvaluator(
    relative_tolerance=1e-18,
).execute(state)

closed_form, generational, fixed_point


In [ ]:
generational_agrees = np.isclose(
    generational.current_amperes,
    closed_form.current_amperes,
    rtol=2e-15,
    atol=0.0,
)
fixed_point_agrees = np.isclose(
    fixed_point.current_amperes,
    closed_form.current_amperes,
    rtol=2e-6,
    atol=0.0,
)
threshold_secondary_emission = 1.0 / (state.avalanche_gain - 1.0)

assert generational_agrees
assert fixed_point_agrees
(
    state.avalanche_gain,
    state.feedback_factor,
    threshold_secondary_emission,
    generational_agrees,
    fixed_point_agrees,
)


In [ ]:
secondary_emission_values = np.linspace(
    0.0,
    0.99 * threshold_secondary_emission,
    300,
    dtype=np.float64,
)
normalized_currents = np.array(
    [
        TownsendClosedFormCurrentEvaluator()
        .execute(
            TownsendDischargeState(
                primary_current_amperes=state.primary_current_amperes,
                first_townsend_coefficient_per_metre=(
                    state.first_townsend_coefficient_per_metre
                ),
                gap_length_metres=state.gap_length_metres,
                secondary_emission_coefficient=float(coefficient),
            )
        )
        .current_amperes
        / state.primary_current_amperes
        for coefficient in secondary_emission_values
    ],
    dtype=np.float64,
)

figure, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(secondary_emission_values, normalized_currents)
axis.axvline(
    threshold_secondary_emission,
    linestyle="--",
    color="black",
    label="represented threshold",
)
axis.set(
    xlabel=r"secondary-emission coefficient $\gamma_e$",
    ylabel="current ratio $i/i_0$",
    title="Townsend current feedback below the represented threshold",
)
axis.grid(True)
axis.legend()
plt.show()


## Interpretation

The three calculation routes agree for the selected sub-threshold state. The current ratio increases as the feedback factor approaches one. At or above the threshold, the evaluators return an infinite current and `converged=False`, matching the represented geometric-feedback model.

## Exercises

1. Set the secondary-emission coefficient to zero and compare $i/i_0$ with the avalanche gain.
2. Vary $\alpha d$ while holding $\gamma_e$ fixed and track the feedback factor.
3. Reduce each iterative route's iteration limit and inspect its result metadata.
